[Guide Index](README.md)

# 05.02 The StarLayer SHACL engine

`starlayer.shacl` is StarLayer's SHACL engine. 

The engine can be used in two ways: through the methods of the `StarShaclProcessor` and through various functions. Most actions of the engine can use both approaches.  The various SHACL notebooks primarily demonstrate the use of the processor.  

This notebook describes the various functions and methods of the engine and where each is discussed in the guide.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.

In [1]:
from starlayer.graph import Namespace
from starlayer.graph import StarLayerGraph
from starlayer.graph.config import set_default_prefixes
from starlayer.shacl import StarShaclProcessor, target_nodes, validate

# EX and SH are starlayer.graph Namespaces: their terms are pyoxigraph NamedNodes, used to look terms up in the results below
# (StarLayerGraphs, see section 2.1); the StarLayerGraph inputs are built by parsing Turtle.
EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")

# Every StarLayerGraph made after this has these prefixes, so results, queries and the report text use short names such as
# ex:alice even for a graph built in Python rather than parsed from Turtle (a prefix the Turtle declares itself still applies).
set_default_prefixes({
    "ex": "http://example.org/",
    "sh": "http://www.w3.org/ns/shacl#",
    "shnex": "http://www.w3.org/ns/shacl-node-expr#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
})


def show(graph):
    """Print a graph as pretty Turtle, declaring only the prefixes it uses."""
    print(graph.serialize(format="turtle", pretty=True, only_used=True))

## 05.02.1 Index of functions and methods

For each operation of the engine, the table shows the function, whether it is also a method of a `StarShaclProcessor` ✅, what it does, and where it is discussed. A method uses the graphs and settings the processor holds. 

Everything in the table, and `StarShaclProcessor`, are imported from `starlayer.shacl`.

| Function | Method | What it does | Discussed in |
|---|:-:|---|---|
| **Validating, evaluating and running rules** | | | |
| `validate()` | ✅ | Checks a data graph against a shapes graph and returns a `ValidationReport`. | [05.01](05.01-validating-with-shacl-1.2.ipynb) |
| `validate_each()` | ✅ | Validates each graph of a dataset against the shapes graph. | [05.01.4.1](05.01-validating-with-shacl-1.2.ipynb) |
| `evaluate()` | ✅ | The values of a node or nodes for the properties of a shape, with `sh:values` calculated and `sh:defaultValue` used, as a graph. | [05h](05h-shacl-evaluate.ipynb) |
| `apply_rules()` | ✅ | Runs the shapes' rules over the data until nothing new is inferred. Returns the inferred triples. The data graph is not changed. | [05c](05c-shacl-inference-rules.ipynb) |
| `target_nodes()` | ✅ | The nodes a shape targets in the data graph, which are the nodes `validate()` checks for that shape. | [05.01.3.6](05.01-validating-with-shacl-1.2.ipynb), [05.03](05.03-shapes-as-classes.ipynb) |
| `target_shapes()` | ✅ | The shapes that target a node, the reverse of `target_nodes()`. | [05.01.3.6](05.01-validating-with-shacl-1.2.ipynb), [05.03](05.03-shapes-as-classes.ipynb) |
| **Writing and extracting shapes** | | | |
| `derive_shape()` |  | Builds one closed node shape from a class or a node in the data graph. Used to start designing a shape from data. | section 05.02.3 |
| `extract_subgraph()` | ✅ | The part of the data graph that one shape covers for a focus node, or that all the shapes of a shapes graph cover. | [05f](05f-shacl-subgraph-extraction.ipynb) |
| `close_shape()` | ✅ | A copy of the shapes graph in which a shape, and the shapes it reaches, are closed. | [05f](05f-shacl-subgraph-extraction.ipynb) |
| `extract_shapes_graph()` | ✅ | One named `sh:ShapesGraph` as a graph of its own, with its members and prefixes. | [05e](05e-shacl-profiling.ipynb) |
| **SHACL UI and editors** | | | |
| `select_widgets()` | ✅ | The widget for each property shape of a node shape, for a given role (editor or viewer). | [05d](05d-shacl-ui.ipynb) |
| `select_widget()` | ✅ | The widget for one property shape and given focus node, by the SHACL UI selection procedure. | [05d](05d-shacl-ui.ipynb) |
| `explain_widgets()` | ✅ | Every candidate widget for a property shape, and the steps that chose the selected widget. | [05d](05d-shacl-ui.ipynb) |
| `candidate_values()` | ✅ | What a form can offer for a field: the values the shape names, with the instances of a class and its subclasses. | [05g](05g-building-on-shacl.ipynb) |
| `value_form()` | ✅ | Whether a field holds one value, an RDF list, or either, read from its shape. | [05g](05g-building-on-shacl.ipynb) |
| `check_value()` | ✅ | Whether a value would be accepted as valid for a field of a node, before it is saved. | [05g](05g-building-on-shacl.ipynb) |
| `form_model()` | ✅ | The fields of a node's form, with their widgets, their values and source of each value. | [05g](05g-building-on-shacl.ipynb) |
| **Checking shapes** | | | |
| `meta_validate()` |  | Checks a shapes graph against SHACL's own shapes, or a proposed edit to a shapes graph, before it is made. | section 05.02.4 |
| `meta_scope()` |  | What a check of an edit would cover, and why. | section 05.02.4.1 |
| **Reading a report** | | | |
| `report_results()` |  | The results of a validation report. | [05g](05g-building-on-shacl.ipynb) |
| `diff_reports()` |  | Compares two validation reports and outputs the changes. | [05g](05g-building-on-shacl.ipynb) |
| `derive_conforms_to()` |  | Derives `sh:conformsToShapesGraph` from a validation report and an optional `specification_graph` | [05e](05e-shacl-profiling.ipynb) |


A function takes the graphs and optional paramters on every call. `StarShaclProcessor` binds a shapes graph, and optionally other graphs and settings, once, so that it can be re-used with different data graphs, and offers most all of the same operations as methods. 

The rest of this guide covers the fucntionality of the SHACL engine not covered elsewhere:

- how the processor is configured, and how a call overrides the configuration (section 05.02.2),
- how to derive a shape from data (section 05.02.3),
- how to check a shapes graph (section 05.02.4), and
- descriptivie shape properties (section 05.02.5).

The functions and the processor take a `StarLayerGraph`, a `StarLayerDataset` or a `pyoxigraph.Store`. Anything else raises a `TypeError` that names the argument; an rdflib graph, for example, has to be converted first with `StarLayerGraph.from_rdflib()`.

The results and errors are classes: `ValidationReport` (the result of `validate()`), `RulesResult` (the result of `apply_rules()`), and the errors `ShapeError`, `RuleError` and `MetaShaclError`, which are all `ValueError`s (see [05.01.4.2](05.01-validating-with-shacl-1.2.ipynb)). `SH` is a namespace of the SHACL terms.

## 05.02.2 Configuring the processor

A `StarShaclProcessor` is configured when it is created. Only the shapes graph is required: every other graph and every setting is optional. Configurations set at creation can be modified. Any configuration can be overridden during an individual method call without changing the configuration. From lowest to highest, the precedence is the built-in default, the processor's configuration, then the call. The functions have no configuration of their own: they take the same settings as keywords on each call.

The processor holds its graphs and does not copy them, so a change made to the data graph or the shapes graph is seen by the next method call. 

The graphs can be given when the processor is created.  Only the shapes graph is required.

| Graph | What it is |
|---|---|
| `shacl_graph` | the shapes graph |
| `ont_graph` | an ontology graph, read when an entailment regime is used ([05.01.4.3](05.01-validating-with-shacl-1.2.ipynb)) |
| `data_graph` | a default data graph, so that `validate()` or `evaluate()` can be called without one |
| `scoring_graph` | the widget scoring catalog for `select_widgets()` (see the [SHACL UI guide](05d-shacl-ui.ipynb)) |

The settings are keywords after the graphs. Each is held by the processor and used by the calls that need it. Related settings are together: checking the shapes graph and its imports, entailment and the class hierarchy, conformance, and limits:

| Setting | Built-in default | What it does | Discussed in |
|---|---|---|---|
| `meta_shacl` | `False` | check the shapes graph against SHACL's own SHACL for shapes when the processor is created, and raise `MetaShaclError` if it does not conform | section 05.02.4 |
| `meta_shapes_extra` | none | additional shapes to be included in the meta_shacl check | section 05.02.4.1 |
| `ignore_imports`, `shapes_graph_loader` | `False`, none | how `owl:imports` of the shapes graph are resolved. A declared import that is not in the graph raises an error unless `shapes_graph_loader` is a function that supplies it or `ignore_imports=True` | section 05.02.2.2 |
| `inference` | none | an entailment regime IRI such as `ENTAILMENT.RDFS`, or several, applied to the data graph of every call | [05.01.4.3](05.01-validating-with-shacl-1.2.ipynb) |
| `recompute` | `"onChange"` | when the entailed triples of `inference` are computed again: `"onChange"` (when the graph has changed), `"never"` or `"always"`, as for a SPARQL query | [05.01.4.3](05.01-validating-with-shacl-1.2.ipynb), [SPARQL entailment](03c-sparql-entailment.ipynb) |
| `engine`, `timeout` | `"hermit"`, none | for the `OWL-Direct` regime only: the reasoner and its time limit, as for a SPARQL query | [05.01.4.3](05.01-validating-with-shacl-1.2.ipynb), [SPARQL entailment](03c-sparql-entailment.ipynb), [OWL DL reasoning](02f-owl-dl-reasoning.ipynb) |
| `subclass_of_in_shapes_graph` | `False` | also read `rdfs:subClassOf` from the shapes graph | [05.03.2](05.03-shapes-as-classes.ipynb) |
| `allow_warnings`, `allow_infos` | `False` | a warning (or an info) no longer makes `conforms` false | [05.01.3.7](05.01-validating-with-shacl-1.2.ipynb) |
| `max_validation_depth` | 500 | the depth limit of nested shape checks; `None` removes it | section 05.02.2.3 |
| `max_iterations`, `max_triples` | 1000, no cap | limits on `apply_rules()`, the method and the function | section 05.02.2.3, [inference rules guide](05c-shacl-inference-rules.ipynb) |

### .2.1 Overriding the configuration

The configuration of a processor is the starting point of each call, not a fixed set of values. A setting has up to three values, and the highest of them is the one a call uses:

1. the built-in default of the setting (see the table above),
2. the processor's configuration: the value given when the processor was created, or set later by assigning the attribute (`myProcessor.allow_warnings = True`), which changes the configuration for every call after that, and
3. the call: a keyword passed to a method of the processor, such as `validate()` or `evaluate()`, which applies to that call only and leaves the processor's configuration as it was.

The same keywords are used at each level. A function has no configuration, so only its built-in defaults and the keywords of the call apply. A method call can also provide a value the processor has none for, such as a data graph when none is bound. This makes a processor a reusable base: it is configured once for what is usual, and a call overrides it for the exception.

Below, `ex:bob` has no name, which is only a warning. The processor is created with `allow_warnings=True` and a bound data graph, so `validate()` needs no data argument and he conforms. A call with `allow_warnings=False` overrides the setting for that call only (does not conform), and the processor's own setting is unchanged afterwards. Assigning `myProcessor.allow_warnings = False` changes the configuration for every call after that (does not conform).

In [2]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ; sh:severity sh:Warning ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:bob a ex:Person .
""", format="turtle")

# the processor is created from the shapes graph with a data graph and a setting, then used with no arguments
myProcessor = StarShaclProcessor(shacl_graph=shapes, data_graph=data, allow_warnings=True)
print("with the processor's setting:", myProcessor.validate().conforms)

# a call can override the setting for that call only
print("strict for this call:", myProcessor.validate(allow_warnings=False).conforms)
print("the processor's setting after the call:", myProcessor.allow_warnings)

# setting the attribute changes the configuration for every call after it
myProcessor.allow_warnings = False
print("after setting the attribute:", myProcessor.validate().conforms)


with the processor's setting: True
strict for this call: False
the processor's setting after the call: True
after setting the attribute: False


### .2.2 Imports of the shapes graph: `ignore_imports` and `shapes_graph_loader`

A shapes graph can import another with `owl:imports`, and SHACL says a processor should follow the imports before it validates. If the processor skipped an import it could not find, any shape that refers to the import would never fire, and the data would seem to conform, so an import that is not in the graph raises an error at the first `validate()`. There are three ways to deal with it: put the imported graph in the shapes graph, give a `shapes_graph_loader` function that supplies it, or set `ignore_imports=True`. The loader is called with the IRI of the import, a `pyoxigraph.NamedNode`, and returns a graph (or `None` when it does not have one). The imported shapes then take part as if they were in the shapes graph. With `ignore_imports=True` the imports are not followed, and the shapes they would have supplied are not used. It is for imports that need no loading: the modules were merged into the shapes graph already, but the `owl:imports` statements were kept, or the import is only a declaration. An import counts as present when its IRI is the subject of triples in the shapes graph. The import of the SHACL vocabulary, `owl:imports sh:`, needs neither a loader nor `ignore_imports`: it is recommended for a graph that uses `sh:ShapeClass` (see [05.03](05.03-shapes-as-classes.ipynb)), and the engine has the vocabulary built in. The [SHACL profiling guide](05e-shacl-profiling.ipynb) covers the `owl:imports` and `sh:ShapesGraph` packaging conventions.

Below, `ex:PersonShape` requires every person to conform to `ex:CommonShape`, which is in a graph that the shapes graph imports. Without a loader, validating raises an error. With a loader that returns the common shapes, `ex:alice`, who has no name, does not conform. With `ignore_imports=True` the common shapes are not used, so she conforms.

In [ ]:
main = StarLayerGraph()
main.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Main a owl:Ontology ; owl:imports <http://example.org/common> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ; sh:node ex:CommonShape .
""", format="turtle")

common = StarLayerGraph()
common.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:CommonShape a sh:NodeShape ; sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

people = StarLayerGraph()
people.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
""", format="turtle")

# the import is not in the shapes graph, and nothing supplies it
try:
    StarShaclProcessor(shacl_graph=main).validate(data_graph=people)
except ValueError as error:
    print("no loader:", error)

# a loader supplies the imported graph for its IRI
def loader(iri):
    return common if iri.value == "http://example.org/common" else None

print("with a loader, conforms:", StarShaclProcessor(shacl_graph=main, shapes_graph_loader=loader).validate(data_graph=people).conforms)

# the imports are not followed
print("with ignore_imports, conforms:", StarShaclProcessor(shacl_graph=main, ignore_imports=True).validate(data_graph=people).conforms)

### .2.3 Limits: `max_validation_depth`, `max_iterations` and `max_triples`

`max_validation_depth` limits how deeply shape checks can nest. A shape that refers to itself, for example a node shape that requires the next node in a chain to conform to the same shape, is checked one level deeper for each node of the chain. A chain deeper than the limit (500 by default) raises a `ShapeError`. `None` removes the limit.

`max_iterations` and `max_triples` limit `apply_rules()`, both the method and the function, which runs the rules until nothing new is inferred. A rule set that has not reached that point after `max_iterations` rounds (1000 by default) raises a `RuleError`, and so does one that has inferred more than `max_triples` triples (no cap by default). The [inference rules guide](05c-shacl-inference-rules.ipynb) covers rules.

Below, a chain of five nodes needs a depth of more than three (with the default it conforms, and with `max_validation_depth=3` it raises a `ShapeError`). A rule that creates a new item for each item never stops, so it ends in a `RuleError` at the limit of five rounds, and again at the limit of ten triples.

In [ ]:
from starlayer.shacl import RuleError, ShapeError

chain_shapes = StarLayerGraph()
chain_shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:NodeShape a sh:NodeShape ;
      sh:targetNode ex:n0 ;
      sh:property [ sh:path ex:next ; sh:node ex:NodeShape ] .
""", format="turtle")
chain = StarLayerGraph()
chain.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:n0 ex:next ex:n1 . ex:n1 ex:next ex:n2 . ex:n2 ex:next ex:n3 . ex:n3 ex:next ex:n4 .
""", format="turtle")

print("default depth, conforms:", StarShaclProcessor(shacl_graph=chain_shapes).validate(data_graph=chain).conforms)
try:
    StarShaclProcessor(shacl_graph=chain_shapes, max_validation_depth=3).validate(data_graph=chain)
except ShapeError as error:
    print("ShapeError:", error)

# every item gets a new item after it, so the rule never stops
endless = StarLayerGraph()
endless.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:Rule a sh:NodeShape ; sh:targetClass ex:Item ;
      sh:rule [ a sh:SPARQLRule ; sh:construct \"\"\"PREFIX ex: <http://example.org/>
        CONSTRUCT { ?next a ex:Item . $this ex:next ?next }
        WHERE { $this a ex:Item . BIND(IRI(CONCAT(STR($this), "x")) AS ?next) }\"\"\" ] .
""", format="turtle")
items = StarLayerGraph()
items.parse(data="@prefix ex: <http://example.org/> . ex:a a ex:Item .", format="turtle")

endlessProcessor = StarShaclProcessor(shacl_graph=endless)
for limit in ({"max_iterations": 5}, {"max_triples": 10}):
    try:
        endlessProcessor.apply_rules(data_graph=items, **limit)
    except RuleError as error:
        print(limit, "RuleError:", error)

## 05.02.3 Deriving shapes from data

`derive_shape()` is a function of `starlayer.shacl`, with no method form, and it is a StarLayer feature, not part of SHACL. Given a data graph and either a class or a node, it writes a shapes graph with one node shape for it. With `cls=`, the shape targets all instances of the class, subclasses included, and its constraints come from what those instances use. With `node=`, the shape targets that one node. The shape is named after its target (`ex:Person` gives `ex:PersonShape`, `ex:alice` gives `ex:AliceShape`) unless `shape=` names it. From the values it finds, it sets the constraints the data supports: `sh:minCount` and `sh:maxCount`, the kind of value (`sh:datatype`, `sh:class` or `sh:nodeKind`) and numeric and string ranges. The shape is `sh:closed`, with `rdf:type` and the usual descriptive properties ignored. It is validated against the same data before it is returned, so it always describes it.

**Why use it.** A shape written from scratch can miss how the data is really used. `derive_shape()` gives a starting point drawn from the data itself, which is useful in three situations:

- **There are no shapes yet.** The derived shape shows what the data contains: which properties, how many values, which kinds of value. It can be read, edited and kept as the first version of the shapes.
- **Later data should look like earlier data.** A shape derived from data that is known to be good can be used to validate data that arrives later, so that a change in its structure shows up as a violation.
- **The shapes should follow the data.** With `template_shape`, a shape that has been edited by hand is brought up to date when the data changes, without losing the constraints the data still satisfies.

A derived shape describes the data it was derived from, not what is intended. For example, `sh:maxCount 1` may only mean that the sample had one value. Review the constraints before relying on them as rules.

Given a `template_shape`, `derive_shape()` reconciles the template's shape for the same class or node with the data: a constraint the data now breaks is recomputed, one that the data cannot fix (such as `sh:pattern`) is dropped with an `rdfs:comment`, and the rest are left alone.

Below, a shape is derived for the class `ex:Person` and another for the node `ex:alice`, and the data is validated against the class shape (conforms).

In [3]:
import warnings

from starlayer.shacl import derive_shape

observed = StarLayerGraph()
observed.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:name "Alice" ; ex:age 30 ; ex:knows ex:bob .
    ex:bob a ex:Person ; ex:name "Bob" ; ex:age 41 .
""", format="turtle")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # the fidelity warning does not matter here
    # a shape for all the instances of a class
    class_shape = derive_shape(observed, cls=EX.Person)
    # a shape for one node
    node_shape = derive_shape(observed, node=EX.alice)
    # the data conforms to the shape derived from it
    conforms = StarShaclProcessor(shacl_graph=class_shape).validate(data_graph=observed).conforms
print("a shape for the class ex:Person")
show(class_shape)
print("a shape for the node ex:alice")
show(node_shape)
print("the data conforms to its own derived shape:", conforms)


a shape for the class ex:Person
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:PersonShape a sh:NodeShape ;
    sh:closed true ;
    sh:ignoredProperties (
        <http://purl.org/dc/terms/created>
        <http://purl.org/dc/terms/modified>
        rdf:type
        rdfs:comment
        rdfs:label
    ) ;
    sh:property [
        a sh:PropertyShape ;
        sh:class ex:Person ;
        sh:maxCount 1 ;
        sh:path ex:knows
    ], [
        a sh:PropertyShape ;
        sh:datatype xsd:integer ;
        sh:maxCount 1 ;
        sh:minCount 1 ;
        sh:path ex:age
    ], [
        a sh:PropertyShape ;
        sh:datatype xsd:string ;
        sh:maxCount 1 ;
        sh:minCount 1 ;
        sh:path ex:name
    ] ;
    sh:targetClass ex:Person .

a shape for the node ex:alice
@prefi

## 05.02.4 Checking a shapes graph: `meta_validate()`

`meta_validate()` checks a shapes graph against SHACL's own shapes, the shapes for shapes. It is what `meta_shacl=True` does when a processor is created, but as a function it needs no processor, so it can check shapes before they are saved or used. A shapes graph that is valid returns `(conforms, report_graph, report_text)`. One that is not valid raises a `MetaShaclError` whose message says which shape and property each problem is about. It holds the lines in `problems`, the full report in `report_text` and the same results as a graph in `report_graph`, so an application can show each problem beside its node. Warnings from the shapes for shapes are tolerated, and `allow_warnings=False` makes them count.

The check also covers things the shapes for shapes cannot say on their own:

- the values a shapes graph gives for the parameters of its constraint components fit the parameter's own constraints, for example text given for an `xsd:integer` parameter,
- the queries it holds (`sh:select`, `sh:ask`, `sh:construct`, `shui:searchQuery`) are valid SPARQL,
- every `sh:pattern` is a regular expression the engine can compile, with the `sh:flags` of the same shape, and
- every member of an `sh:languageIn` list is a well-formed language range.

Each is a result in the report like any other ([5.g](05g-building-on-shacl.ipynb)). A term in the SHACL namespace that SHACL does not define, such as `sh:minCountt`, is reported as a warning. Terms in other namespaces are never reported, and shapes are not closed.

Below, a valid shapes graph passes, and a shapes graph with two `sh:datatype` values on one property shape raises a `MetaShaclError`.

In [ ]:
from starlayer.shacl import MetaShaclError, meta_validate

valid = StarLayerGraph()
valid.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")
conforms, report_graph, report_text = meta_validate(valid)
print("valid shapes:", conforms)

two_datatypes = StarLayerGraph()
two_datatypes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:datatype xsd:string, xsd:integer ] .
""", format="turtle")
try:
    meta_validate(two_datatypes)
except MetaShaclError as error:
    print("MetaShaclError:", error)
    print(error.problems)

### .4.1 Adding your own checks: `meta_shapes_extra`

`meta_shapes_extra` gives a processor with `meta_shacl=True` additional shapes to check the shapes graph against, along with SHACL's own. They are ordinary shapes whose data is the shapes graph, so a rule about how shapes are written is a shape that targets `sh:NodeShape` or `sh:PropertyShape`. This is a way to apply a house style, such as requiring every node shape to have a label, when shapes are created. The same graphs can be passed to `meta_validate()` as `extra_graphs`.

Below, a shape requires every node shape to have an `rdfs:label`. A shapes graph without labels passes SHACL's own check but fails this one, and one with a label passes.

In [ ]:
extra = StarLayerGraph()
extra.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:LabelRule a sh:NodeShape ;
      sh:targetClass sh:NodeShape ;
      sh:property [ sh:path rdfs:label ; sh:minCount 1 ; sh:message "A node shape needs an rdfs:label" ] .
""", format="turtle")

unlabelled = StarLayerGraph()
unlabelled.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person .
""", format="turtle")

labelled = StarLayerGraph()
labelled.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:PersonShape a sh:NodeShape ; rdfs:label "Person" ; sh:targetClass ex:Person .
""", format="turtle")

StarShaclProcessor(shacl_graph=unlabelled, meta_shacl=True)   # SHACL's own shapes accept it
try:
    StarShaclProcessor(shacl_graph=unlabelled, meta_shacl=True, meta_shapes_extra=[extra])
except MetaShaclError as error:
    print("MetaShaclError:", error)
StarShaclProcessor(shacl_graph=labelled, meta_shacl=True, meta_shapes_extra=[extra])
print("with a label: accepted")

### 05.02.4.1 Checking an edit before it is made

An editor wants to know whether an edit would leave the shapes graph valid before it makes the edit. `meta_validate(shapes_graph, add=..., remove=...)` takes the triples of the proposed edit and checks the graph as it would be after them. The graph itself is not changed. Only the nodes the edit touches are checked, together with the shapes that refer to them, and every other node of the graph stays visible to the check, so a shape's type, the prefixes of a query and the parameters of a constraint component are still read from the whole graph. On a large shapes graph that takes a fraction of a second where the full check takes seconds. `nodes=[...]` checks given nodes of the current graph, and `with_dependents=False` leaves out the shapes that refer to them.

Some edits can change what other nodes mean: a prefix declaration, a constraint component and its parameters and validators, a rule template, and the type of a node that already exists. These are risky, and the check then covers the whole graph. A new node, with the shapes and properties that come with it, is not risky. `meta_scope()` gives the decision without running the check: a `MetaScope` with `full`, the `nodes` that are covered, and the `reasons`. A `MetaShaclError` carries the scope it used in `scope`. A check of part of the graph is for the moment between edits; run the full check before the shapes are saved or published.

In [ ]:
from starlayer.graph import Namespace, StarLayerGraph
from starlayer.shacl import SH, MetaShaclError, meta_scope, meta_validate

EX = Namespace("http://example.org/")
XSD = Namespace("http://www.w3.org/2001/XMLSchema#")
current = StarLayerGraph()
current.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ; sh:property ex:AgeShape .
    ex:AgeShape sh:path ex:age ; sh:datatype xsd:integer .
""", format="turtle")

second_datatype = [(EX.AgeShape, SH.datatype, XSD.string)]  # an edit not yet made
try:
    meta_validate(current, add=second_datatype)
except MetaShaclError as error:
    print("edit refused:", error.scope.reasons)
print("the graph is unchanged:", len(list(current.triples((EX.AgeShape, SH.datatype, None)))), "datatype")

prefix_edit = [(EX.PersonShape, SH.declare, EX.Prefixes)]
print("full check for the datatype edit:", meta_scope(current, add=second_datatype).full)
print("a prefix edit:", meta_scope(current, add=prefix_edit).reasons)

edit refused: ('checked 3 nodes: 2 changed, 1 dependent',)
the graph is unchanged: 1 datatype
full check for the datatype edit: False
a prefix edit: ('full check: sh:declare changed on ex:PersonShape',)


## 05.02.5 Properties that describe a shape but do not validate

SHACL 1.2 adds various properties that tools can read but a validating processor ignores. They describe a shape and the kind of thing it is about, so a shape that is also a class (see [05.03](05.03-shapes-as-classes.ipynb)) can carry the description of the class and of its constraints together.

- `sh:name` and `sh:description` name and describe the shape's defining characteristic: the path of a property shape, or the kind of resource a node shape targets.
- `sh:intent` states in words a rule the shape is meant to express, including rules the shape cannot check.
- `rdfs:label` and `rdfs:comment` describe the shape resource itself. Note that for a `sh:ShapeClass`, these are properties of the class as well.
- `sh:unit` gives the unit of measure or currency of the values, as a UCUM or ISO 4217 code or an IRI from a units vocabulary.
- `sh:agentInstruction` is a text instruction for a software agent that reads the shape.
- `sh:codeIdentifier` suggests a name for the shape in generated code or APIs (letters, digits and underscores, not starting with a digit), and `sh:codeType` the datatype that code should receive for each value.
- `sh:order` now accepts an `xsd:integer` as well as an `xsd:decimal` to sort property shapes in a form.


Below, a node shape and a property shape use these properties. Only `sh:minCount`, `sh:datatype` and `sh:minInclusive` validate, so the shapes check the same data with or without the others.


```turtle
@prefix ex:   <http://example.org/> .
@prefix sh:   <http://www.w3.org/ns/shacl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

ex:PersonShape a sh:NodeShape ;
    sh:name "Person" ;                                  # the kind of resource the shape describes
    sh:description "A living individual that the organisation keeps records about." ;
    rdfs:label "Person shape" ;                         # the shape resource itself
    rdfs:comment "The shape that checks the data held about a person." ;
    sh:intent "A person's data is kept for as long as they are a customer." ;
    sh:codeIdentifier "Person" ;
    sh:targetClass ex:Person ;
    sh:property ex:Person-weight .

ex:Person-weight a sh:PropertyShape ;
    sh:name "Weight" ;                                  # the property path, ex:weight
    sh:description "The person's body weight." ;
    sh:intent "Each person has a weight." {| sh:formalized true |} ;
    sh:intent "The weight is at least 20 kg." {| sh:formalized true |} ;
    sh:intent "The weight is measured, not estimated." ;   # a rule the shape cannot check
    sh:order 2 ;
    sh:unit "kg" ;
    sh:codeIdentifier "weight" ;
    sh:codeType xsd:decimal ;
    sh:agentInstruction "Do not estimate a weight that is missing. Ask for it." ;
    sh:path ex:weight ;
    sh:minCount 1 ;
    sh:datatype xsd:decimal ;
    sh:minInclusive 20 .
```